# CS-781 Phase 5: P5_C2_legalbert_full

**Architecture:** `nlpaueb/legal-bert-base-uncased`; **adaptation:** `full`. This notebook is standalone and tuned for one Colab **T4 (16 GB)** runtime. It downloads the official gated Phase 5 files, trains on the official train split, selects a threshold on dev Macro-F1, and writes `submission.zip` to Google Drive.

Run cells top to bottom. Add a Hugging Face **read** token under Colab Secrets as `HF_TOKEN` and grant notebook access, or enter it in the hidden prompt. The account must already have access to the official dataset. Each notebook needs its **own Colab GPU runtime**. Training is time-capped for a four-hour deadline. It saves an early usable ZIP to Drive, then replaces it only if a later development score improves. The only ZIP member is root-level `prediction.json`.

**Course rule warning:** this public pretrained encoder is not one of your original Phase 1 checkpoints. Use these runs only if course rules allow external pretrained weights. The local handoff lacks the four original Phase 1 checkpoints.


In [ ]:
%pip -q install "transformers==4.57.1" "peft==0.17.1" "huggingface_hub==0.36.0" "sentencepiece>=0.2.0" "protobuf>=4.25,<7"


In [ ]:
RUN_ID = 'P5_C2_legalbert_full'
MODEL_ID = 'nlpaueb/legal-bert-base-uncased'
METHOD = 'full'
MAX_LENGTH = 512
TRAIN_BATCH = 8
EVAL_BATCH = 16
GRAD_ACCUM = 2
EPOCHS = 2
TRAIN_LIMIT = None
LEARNING_RATE = 2e-05
FIRST_EXPORT_MINUTES = 20
TRAIN_BUDGET_MINUTES = 100
RESERVED_INFERENCE_MINUTES = 40

import gc, hashlib, json, math, os, random, shutil, time, zipfile
from pathlib import Path
from getpass import getpass
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from huggingface_hub import hf_hub_download, model_info
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from google.colab import drive, userdata

assert torch.cuda.is_available(), "Colab Runtime > Change runtime type > T4 GPU is required"
print("GPU:", torch.cuda.get_device_name(0))
assert torch.cuda.get_device_properties(0).total_memory >= 14_000_000_000, "This notebook is tuned for a 16 GB T4"
random.seed(42); np.random.seed(42); torch.manual_seed(42); torch.cuda.manual_seed_all(42)
torch.backends.cuda.matmul.allow_tf32 = True
ROOT = Path("/content/cs781_phase5")
ROOT.mkdir(parents=True, exist_ok=True)
try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None
if not HF_TOKEN:
    HF_TOKEN = getpass("Enter a Hugging Face read token from your approved account (hidden): ").strip()
assert HF_TOKEN, "A Hugging Face token is required for the official gated dataset"
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/CS781_PHASE5_20261028")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("Output folder:", DRIVE_ROOT / RUN_ID)


In [ ]:
DATASET_ID = "Exploration-Lab/CS-781-Capstone"
DATASET_REVISION = "93b0059be756c6c461f3a225de25aabda7d5f84a"
def load_split(split):
    name = ("Test/" if split == "test" else "Train/") + split + ".json"
    path = hf_hub_download(repo_id=DATASET_ID, repo_type="dataset",
                           filename="Phase_5_LSI/" + name,
                           revision=DATASET_REVISION, token=HF_TOKEN)
    with open(path, encoding="utf-8") as stream:
        raw = json.load(stream)
    assert isinstance(raw, list) and raw
    output = []
    for row in raw:
        assert isinstance(row["id"], str) and isinstance(row["text"], str)
        item = {"id": row["id"], "text": row["text"]}
        if split != "test":
            values = row["labels"]
            assert values and all(type(label) is int and label >= 0 for label in values)
            item["labels"] = list(dict.fromkeys(values))
        output.append(item)
    assert len({row["id"] for row in output}) == len(output)
    return output

rows = {split: load_split(split) for split in ("train", "dev", "test")}
assert not ({r["id"] for r in rows["train"]} & {r["id"] for r in rows["dev"]})
assert not ({r["id"] for r in rows["train"]} & {r["id"] for r in rows["test"]})
assert not ({r["id"] for r in rows["dev"]} & {r["id"] for r in rows["test"]})
label_ids = sorted({int(label) for split in ("train", "dev")
                    for row in rows[split] for label in row["labels"]})
assert label_ids == list(range(100)), f"Unexpected statute IDs: {label_ids}"
label_index = {label: index for index, label in enumerate(label_ids)}
print({name: len(split) for name, split in rows.items()}, "labels:", len(label_ids))


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
assert tokenizer.pad_token_id is not None
SPECIAL = tokenizer.num_special_tokens_to_add(pair=False)
BODY = MAX_LENGTH - SPECIAL

def encode_case(text):
    if len(text) > 24000:
        left = tokenizer.encode(text[:12000], add_special_tokens=False)[:BODY // 2]
        right = tokenizer.encode(text[-12000:], add_special_tokens=False)[-(BODY - len(left)):]
        tokens = left + right
    else:
        tokens = tokenizer.encode(text, add_special_tokens=False)
        if len(tokens) > BODY:
            left = BODY // 2
            tokens = tokens[:left] + tokens[-(BODY - left):]
    encoded = tokenizer.prepare_for_model(tokens, add_special_tokens=True,
                                           truncation=True, max_length=MAX_LENGTH)
    return {key: value for key, value in encoded.items()
            if key in ("input_ids", "attention_mask", "token_type_ids")}

class CaseDataset(Dataset):
    def __init__(self, source, labeled):
        self.rows = source
        self.encoded = [encode_case(row["text"]) for row in source]
        self.labels = None
        if labeled:
            matrix = np.zeros((len(source), len(label_ids)), dtype=np.float32)
            for i, row in enumerate(source):
                matrix[i, [label_index[x] for x in row["labels"]]] = 1
            self.labels = matrix
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        item = dict(self.encoded[index])
        if self.labels is not None:
            item["labels"] = self.labels[index]
        return item

def collate(batch):
    labels = [item.pop("labels") for item in batch] if "labels" in batch[0] else None
    result = tokenizer.pad(batch, padding=True, pad_to_multiple_of=8,
                           return_tensors="pt")
    if METHOD == "longformer_full":
        global_mask = torch.zeros_like(result["input_ids"])
        global_mask[:, 0] = 1
        result["global_attention_mask"] = global_mask
    if labels is not None:
        result["labels"] = torch.tensor(np.stack(labels), dtype=torch.float32)
    return result

train_rows = rows["train"]
if TRAIN_LIMIT is not None:
    selection = np.random.default_rng(42).permutation(len(train_rows))[:TRAIN_LIMIT]
    train_rows = [train_rows[int(i)] for i in selection]
train_data = CaseDataset(train_rows, True)
dev_data = CaseDataset(rows["dev"], True)
test_data = CaseDataset(rows["test"], False)
train_loader = DataLoader(train_data, batch_size=TRAIN_BATCH, shuffle=True,
                          collate_fn=collate, num_workers=2, pin_memory=True)
dev_loader = DataLoader(dev_data, batch_size=EVAL_BATCH, shuffle=False,
                        collate_fn=collate, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_data, batch_size=EVAL_BATCH, shuffle=False,
                         collate_fn=collate, num_workers=2, pin_memory=True)
print("Tokenized:", len(train_data), len(dev_data), len(test_data))


In [ ]:
model_revision = model_info(MODEL_ID).sha
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, num_labels=len(label_ids), problem_type="multi_label_classification",
    ignore_mismatched_sizes=True)
model.config.use_cache = False
if METHOD != "deberta_full":
    model.gradient_checkpointing_enable()
if METHOD == "lora":
    from peft import LoraConfig, TaskType, get_peft_model
    config = LoraConfig(task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16,
                        lora_dropout=0.05, target_modules=["query", "value"],
                        modules_to_save=["classifier"])
    model = get_peft_model(model, config)
    if hasattr(model, "enable_input_require_grads"):
        model.enable_input_require_grads()
    model.print_trainable_parameters()
model.to("cuda")
counts = train_data.labels.sum(axis=0).clip(min=1)
positive_weights = np.clip((len(train_data) - counts) / counts, 1, 20)
positive_weights = torch.tensor(positive_weights, device="cuda", dtype=torch.float32)
optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad),
                               lr=LEARNING_RATE, weight_decay=0.01)
updates = math.ceil(len(train_loader) / GRAD_ACCUM) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=max(1, int(updates * 0.06)),
    num_training_steps=updates)
scaler = torch.amp.GradScaler("cuda")

def score_thresholds(probabilities, truth):
    best = (-1.0, 0.0)
    truth = truth.astype(bool)
    for threshold in np.arange(0.05, 0.701, 0.025):
        pred = probabilities >= threshold
        tp = (pred & truth).sum(axis=0)
        fp = (pred & ~truth).sum(axis=0)
        fn = (~pred & truth).sum(axis=0)
        denominator = 2 * tp + fp + fn
        f1 = np.divide(2 * tp, denominator,
                       out=np.zeros(len(label_ids), dtype=np.float64),
                       where=denominator != 0).mean()
        best = max(best, (float(f1), float(threshold)))
    return best

@torch.inference_mode()
def predict(loader):
    model.eval()
    chunks = []
    for batch in loader:
        batch.pop("labels", None)
        batch = {key: value.to("cuda", non_blocking=True) for key, value in batch.items()}
        with torch.amp.autocast("cuda", dtype=torch.float16):
            logits = model(**batch).logits
        chunks.append(torch.sigmoid(logits.float()).cpu().numpy())
    return np.concatenate(chunks, axis=0)

output = DRIVE_ROOT / RUN_ID
output.mkdir(parents=True, exist_ok=True)
local_output = ROOT / RUN_ID
local_output.mkdir(parents=True, exist_ok=True)
history = []
best_score, best_threshold = -1.0, 0.0
started = time.monotonic()
last_export_step = 0
total_batches = 0

def copy_atomic(source, target):
    temporary = target.with_name(target.name + ".uploading")
    shutil.copy2(source, temporary)
    os.replace(temporary, target)

def export_current(stage, epoch, batch_count, train_loss):
    global best_score, best_threshold, last_export_step
    print(f"Validating {stage} after {batch_count} training batches", flush=True)
    dev_probs = predict(dev_loader)
    score, threshold = score_thresholds(dev_probs, dev_data.labels)
    record = {"stage": stage, "epoch": epoch, "batches": batch_count,
              "train_loss": train_loss, "dev_macro_f1": score,
              "threshold": threshold,
              "elapsed_minutes": round((time.monotonic() - started) / 60, 1)}
    history.append(record)
    print(record, flush=True)
    if score <= best_score and (output / "submission.zip").exists():
        print("Earlier submission remains better; keeping it", flush=True)
        return
    test_probs = predict(test_loader)
    assert test_probs.shape == (len(rows["test"]), len(label_ids))
    assert np.isfinite(test_probs).all()
    predictions = [{"id": row["id"],
                    "pred": [int(label) for label, probability in zip(label_ids, prob)
                             if probability >= threshold]}
                   for row, prob in zip(rows["test"], test_probs)]
    assert len(predictions) == 13019
    assert [p["id"] for p in predictions] == [r["id"] for r in rows["test"]]
    assert all(len(p["pred"]) == len(set(p["pred"])) for p in predictions)
    prediction_path = local_output / "prediction.json"
    prediction_path.write_text(json.dumps(predictions, separators=(",", ":")) + "\n",
                               encoding="utf-8")
    zip_path = local_output / "submission.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED, compresslevel=9) as archive:
        archive.write(prediction_path, "prediction.json")
    with zipfile.ZipFile(zip_path) as archive:
        assert archive.namelist() == ["prediction.json"]
        assert archive.testzip() is None
    np.savez_compressed(local_output / "probabilities.npz",
                        dev=dev_probs, test=test_probs,
                        labels=np.asarray(label_ids),
                        dev_ids=np.asarray([r["id"] for r in rows["dev"]]),
                        test_ids=np.asarray([r["id"] for r in rows["test"]]))
    manifest = {"run_id": RUN_ID, "model_id": MODEL_ID,
                "model_revision": model_revision, "method": METHOD,
                "dataset_id": DATASET_ID, "dataset_revision": DATASET_REVISION,
                "train_cases_available": len(train_data), "batches_completed": batch_count,
                "dev_cases": len(dev_data), "test_cases": len(test_data),
                "max_length": MAX_LENGTH, "train_batch": TRAIN_BATCH,
                "gradient_accumulation": GRAD_ACCUM, "learning_rate": LEARNING_RATE,
                "dev_macro_f1": score, "threshold": threshold,
                "history": history, "stage": stage,
                "zip_sha256": hashlib.sha256(zip_path.read_bytes()).hexdigest()}
    manifest_path = local_output / "manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    copy_atomic(prediction_path, output / "prediction.json")
    copy_atomic(zip_path, output / "submission.zip")
    copy_atomic(local_output / "probabilities.npz", output / "probabilities.npz")
    copy_atomic(manifest_path, output / "manifest.json")
    try:
        checkpoint_path = local_output / "checkpoint.pt"
        if METHOD == "lora":
            trainable_state = {name: parameter.detach().cpu()
                               for name, parameter in model.named_parameters()
                               if parameter.requires_grad}
            torch.save({"trainable_state": trainable_state,
                        "model_id": MODEL_ID, "method": METHOD}, checkpoint_path)
        else:
            torch.save({"model_state": model.state_dict(),
                        "model_id": MODEL_ID, "method": METHOD}, checkpoint_path)
        copy_atomic(checkpoint_path, output / "checkpoint.pt")
    except Exception as exc:
        print("ZIP is safe on Drive; checkpoint copy failed:", repr(exc), flush=True)
    best_score, best_threshold, last_export_step = score, threshold, batch_count
    print("READY ON DRIVE:", output / "submission.zip", "dev Macro-F1:", score, flush=True)

stop_training = False
for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    running_loss = 0.0
    for step, batch in enumerate(train_loader, start=1):
        targets = batch.pop("labels").to("cuda", non_blocking=True)
        inputs = {key: value.to("cuda", non_blocking=True) for key, value in batch.items()}
        with torch.amp.autocast("cuda", dtype=torch.float16):
            logits = model(**inputs).logits.float()
            loss = torch.nn.functional.binary_cross_entropy_with_logits(
                logits, targets, pos_weight=positive_weights)
        running_loss += float(loss.detach())
        scaler.scale(loss / GRAD_ACCUM).backward()
        total_batches += 1
        if step % GRAD_ACCUM == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad), 1.0)
            scaler.step(optimizer); scaler.update(); scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            elapsed = (time.monotonic() - started) / 60
            if last_export_step == 0 and elapsed >= FIRST_EXPORT_MINUTES:
                export_current("early", epoch, total_batches, running_loss / step)
                model.train()
            if elapsed >= TRAIN_BUDGET_MINUTES:
                stop_training = True
                break
        if step % 200 == 0:
            print(f"epoch {epoch} batch {step}/{len(train_loader)} loss {running_loss/step:.4f}", flush=True)
    if stop_training:
        break
    if last_export_step == 0:
        export_current("epoch", epoch, total_batches, running_loss / len(train_loader))
    model.train()

elapsed = (time.monotonic() - started) / 60
if last_export_step == 0 or (total_batches > last_export_step and
                            elapsed <= 175 - RESERVED_INFERENCE_MINUTES):
    export_current("final", epoch, total_batches, running_loss / max(1, step))
print("Best saved ZIP:", output / "submission.zip", "dev Macro-F1:", best_score)


In [ ]:
zip_path = DRIVE_ROOT / RUN_ID / "submission.zip"
assert zip_path.exists(), "No ZIP was saved; inspect the training cell output"
with zipfile.ZipFile(zip_path) as archive:
    assert archive.namelist() == ["prediction.json"]
    assert archive.testzip() is None
print("READY:", zip_path)
print("Dev Macro-F1:", best_score)


## Submit

Upload `submission.zip` from the printed Drive folder to the Phase 5 Codabench page. Record the Codabench submission ID and score; creating the ZIP does not upload it.
